# Notebook 01 — Exploratory Data Analysis

**ECE 228 SP26 · Akshat Gupta**

Explores the 619,705-window Starlink passive detection dataset (Feb–Mar 2026, UCSD rooftop). Goal: understand distributions, class structure, and feature informativeness before training.

In [ ]:
import sys, json
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from gp_detector.dataset import load_raw, engineer_features, make_labeled_split, FEATURE_COLS

sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})


## 1. Dataset overview

In [ ]:
raw = load_raw()
print(f"Total rows : {len(raw):,}")
print(f"Columns    : {list(raw.columns)}")
print()
conf_counts = raw['confidence'].value_counts()
for cls, n in conf_counts.items():
    print(f"  {cls:12s}  {n:7,}  ({100*n/len(raw):.1f}%)")


In [ ]:
ts = pd.to_datetime(raw['timestamp_utc'])
print(f"Campaign : {ts.min().date()}  to  {ts.max().date()}  ({(ts.max()-ts.min()).days} days)")
print(f"Unique satellites identified: {raw['sat_name'].nunique():,}")
print(f"Elevation range: {raw['sat_elevation'].min():.1f}° – {raw['sat_elevation'].max():.1f}°")


## 2. Feature engineering

In [ ]:
df = engineer_features(raw)
print(f"{'Feature':<20} {'min':>8} {'max':>8} {'mean':>8}")
print('-'*48)
for col in FEATURE_COLS:
    print(f"{col:<20} {df[col].min():>8.3f} {df[col].max():>8.3f} {df[col].mean():>8.3f}")


## 3. z-score distribution by class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = {'BACKGROUND': '#7f8c8d', 'SOFT': '#f39c12', 'HARD': '#e74c3c'}

for conf, color in colors.items():
    vals = df[df.confidence == conf]['z_score']
    axes[0].hist(vals, bins=80, alpha=0.6, color=color,
                 label=f'{conf} (n={len(vals):,})', density=True)
axes[0].axvline(3.0, ls=':', color='black', lw=1.2, label='z=3 soft boundary')
axes[0].axvline(4.0, ls='--', color='#c0392b', lw=1.5, label='z=4 hard threshold')
axes[0].set(xlabel='z-score', ylabel='Density', title='z-score Distribution by Class', xlim=(-1,7))
axes[0].legend(fontsize=9)

for conf, color in colors.items():
    vals = df[df.confidence == conf]['z_score']
    axes[1].hist(vals.clip(2,5), bins=60, alpha=0.6, color=color, label=conf, density=True)
axes[1].axvline(3.0, ls=':', color='black', lw=1.2)
axes[1].axvline(4.0, ls='--', color='#c0392b', lw=1.5)
axes[1].set(xlabel='z-score (clipped 2–5)', title='Zoomed: Decision Boundary Region')
axes[1].legend(fontsize=9)

for ax in axes: ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../results/eda_z_distribution.png', bbox_inches='tight')
plt.show()


## 4. Elevation vs z-score

Higher elevation = shorter path, less attenuation, better dish alignment. The physics predicts a positive correlation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
rng = np.random.default_rng(42)

for conf, color, size, alpha in [('BACKGROUND','#aab7b8',3,0.12),('SOFT','#f39c12',7,0.4),('HARD','#e74c3c',14,0.8)]:
    sub = df[df.confidence == conf]
    idx = rng.choice(len(sub), min(3000, len(sub)), replace=False)
    axes[0].scatter(sub['sat_elevation'].values[idx], sub['z_score'].values[idx],
                    s=size, alpha=alpha, color=color, linewidths=0, label=conf)
axes[0].axhline(4.0, ls='--', lw=1.3, color='#c0392b', label='z=4 threshold')
axes[0].set(xlabel='Satellite Elevation (°)', ylabel='z-score', title='z-score vs Elevation')
axes[0].legend(fontsize=9)

bins = np.arange(50, 92, 5)
for conf, color in [('BACKGROUND','#7f8c8d'),('SOFT','#f39c12'),('HARD','#e74c3c')]:
    sub = df[df.confidence == conf]
    means = [sub[(sub.sat_elevation>=lo)&(sub.sat_elevation<lo+5)]['z_score'].mean() for lo in bins]
    axes[1].plot(bins+2.5, means, 'o-', color=color, label=conf, lw=1.8, ms=5)
axes[1].set(xlabel='Elevation (°)', ylabel='Mean z-score', title='Mean z-score per 5° Elevation Bin')
axes[1].legend(fontsize=9)

for ax in axes: ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../results/eda_elevation.png', bbox_inches='tight')
plt.show()

hard_bg = df[df.confidence.isin(['BACKGROUND','HARD'])]
corr = hard_bg[['sat_elevation','z_score']].corr().iloc[0,1]
print(f"Pearson r(elevation, z_score) on HARD+BG: {corr:.4f}")


## 5. Harmonics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
offsets = {'BACKGROUND': -0.25, 'SOFT': 0, 'HARD': 0.25}

for conf, color in colors.items():
    sub = df[df.confidence==conf]
    dist = sub['harmonics'].value_counts(normalize=True).sort_index()
    axes[0].bar(dist.index + offsets[conf], dist.values, width=0.25, color=color, alpha=0.8, label=conf)
axes[0].set(xlabel='Harmonic Count', ylabel='Fraction of class', title='Harmonic Distribution by Class')
axes[0].set_xticks([0,1,2,3]); axes[0].legend(fontsize=9)

harm_vals = [0,1,2,3]
hard_rates = [(df[df.harmonics==h].confidence=='HARD').mean() for h in harm_vals]
soft_rates = [(df[df.harmonics==h].confidence=='SOFT').mean() for h in harm_vals]
x = np.array(harm_vals)
axes[1].bar(x-0.2, hard_rates, width=0.35, color='#e74c3c', alpha=0.8, label='HARD rate')
axes[1].bar(x+0.2, soft_rates, width=0.35, color='#f39c12', alpha=0.8, label='SOFT rate')
axes[1].set(xlabel='Harmonic Count', ylabel='Detection rate', title='HARD/SOFT Rate by Harmonic Count')
axes[1].set_xticks(harm_vals); axes[1].legend(fontsize=9)

for ax in axes: ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../results/eda_harmonics.png', bbox_inches='tight')
plt.show()


## 6. Diurnal patterns

In [ ]:
ts = pd.to_datetime(df['timestamp_utc'])
df['hour'] = ts.dt.hour + ts.dt.minute / 60.0

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for conf, color in colors.items():
    sub = df[df.confidence==conf]
    hourly = sub.groupby(sub['hour'].astype(int))['z_score'].mean()
    axes[0].plot(hourly.index, hourly.values, 'o-', color=color, label=conf, lw=1.8, ms=5)
axes[0].set(xlabel='Hour of Day (UTC)', ylabel='Mean z-score', title='Diurnal z-score Pattern', xlim=(0,23))
axes[0].legend(fontsize=9)

hard_by_hour = df[df.confidence=='HARD'].groupby(df['hour'].astype(int)).size()
total_by_hour = df.groupby(df['hour'].astype(int)).size()
rate = (hard_by_hour / total_by_hour).fillna(0)
axes[1].bar(rate.index, rate.values, color='#e74c3c', alpha=0.8)
axes[1].set(xlabel='Hour of Day (UTC)', ylabel='HARD detection rate', title='HARD Detection Rate by Hour')

for ax in axes: ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../results/eda_diurnal.png', bbox_inches='tight')
plt.show()


## 7. Feature correlation matrix

In [ ]:
train_df, soft_df = make_labeled_split(df, seed=42)
corr = train_df[FEATURE_COLS + ['label']].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, annot_kws={'size': 9})
ax.set_title('Feature Correlation Matrix (HARD vs BACKGROUND train set)')
plt.tight_layout()
plt.savefig('../results/eda_correlation.png', bbox_inches='tight')
plt.show()


## 8. Key takeaways

1. **z_score dominates** — HARD and BACKGROUND are cleanly separated. SOFT sits in the ambiguous z ∈ [3,4] band.
2. **Elevation matters** — positive correlation with z_score across all classes. Path length + beam geometry is real.
3. **Harmonics are informative** — higher harmonic count strongly predicts HARD detection.
4. **Diurnal variation is real** — 20% swing in detection rate across the day; hour features are worth including.
5. **The ML challenge is calibration, not discrimination** — AUROC will be ~1.0 for all models. The interesting question is whether models assign meaningful probabilities to SOFT windows.